In [12]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
import math
from IPython.display import Audio

# Define the RK4 solver
def rk4_step(f, u, t, dt, *args):
    k1 = f(u, t, *args)
    k2 = f(u + 0.5*dt*k1, t + 0.5*dt, *args)
    k3 = f(u + 0.5*dt*k2, t + 0.5*dt, *args)
    k4 = f(u + dt*k3, t + dt, *args)
    return u + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

# Define the Lorenz system
sigma = 10.0
rho = 28.0
beta = 8.0/3.0
def lorenz(u, t, sigma=sigma, rho=rho, beta=beta):
    x, y, z = u
    return np.array([sigma*(y - x),
                     x*(rho - z) - y,
                     x*y - beta*z], dtype=float)

# Jacobian entries for single Lorenz vector field at state u=(x,y,z)
def jacobian_lorenz_single(u, sigma=sigma, rho=rho, beta=beta):
    x, y, z = u
    return np.array([
        [-sigma,      sigma,      0.0],
        [rho - z,     -1.0,      -x  ],
        [y,           x,        -beta]
    ], dtype=float)

# We need a Jacobian for the ERROR-system linearization using both systems.
# This J should be used when integrating dV/dt = J(t) V for the variational equation.
def jacobian_error_mixed(tx_state, rx_state, k, sigma=sigma, rho=rho, beta=beta):
    x1, y1, z1 = tx_state
    x2, y2, z2 = rx_state
    J = np.array([
        [-(sigma + k), sigma, 0.0],
        [rho - z1, -1.0, -x2],
        [y2, x1,   -beta]
    ], dtype=float)
    return J


In [13]:
def load_multiple_speechcommands_messages(
        label=None,
        target_rate=200,
        repeat_to_len=None,
        num_samples=5,
    ):
    """
    Load multiple samples from Speech Commands.
    For each sample:
        - normalize
        - downsample to target_rate
        - repeat-to-length if needed
    Filters out '_unknown_' labels by default.
    Returns:
        audios_ds: list of numpy arrays after processing
        labels: list of labels
        sr: original sample rate
        factor: downsample factor
    """
    ds, info = tfds.load("speech_commands", split="train", with_info=True, shuffle_files=True)
    sr = info.features["audio"].sample_rate  # 16000

    audios_ds = []
    labels = []
    for ex in ds:
        audio = ex["audio"].numpy().astype(np.float32)
        lab = info.features["label"].int2str(ex["label"].numpy())

        if (lab not in ["_unknown_"]) and (label is None or lab == label):
            audio = audio / np.max(np.abs(audio))

            factor = int(sr // target_rate)
            factor = max(factor, 1)
            m_ds = audio[::factor]

            if repeat_to_len is not None and len(m_ds) < repeat_to_len:
                nrep = int(np.ceil(repeat_to_len / len(m_ds)))
                m_ds = np.tile(m_ds, nrep)[:repeat_to_len]

            audios_ds.append(m_ds.astype(np.float32))
            labels.append(lab)

            if len(audios_ds) >= num_samples:
                break

    if len(audios_ds) == 0:
        raise RuntimeError("No matching audio found!")

    return audios_ds, labels, sr, factor


In [ ]:
def simulate_tx(u0_tx, t_steps, dt):
    traj = np.zeros((t_steps, 3), dtype=float)
    u = u0_tx.copy().astype(float)
    t = 0.0
    for i in range(t_steps):
        traj[i] = u.copy()
        u = rk4_step(lorenz, u, t, dt)
        t += dt
    return traj

def simulate_tx_rx_with_message(u0_tx, u0_rx, m_array, alpha, k, dt):

    N = len(m_array)
    tx = np.zeros((N,3), dtype=float)
    rx = np.zeros((N,3), dtype=float)
    s  = np.zeros(N, dtype=float)

    u_tx = u0_tx.copy().astype(float)
    u_rx = u0_rx.copy().astype(float)
    t = 0.0

    for i in range(N):
        u_tx_next = rk4_step(lorenz, u_tx, t, dt)
        s_t = u_tx_next[0] + alpha * m_array[i]
        def rx_rhs(u_local, t_local, s_val=s_t):
            x, y, z = u_local
            dx = sigma*(y - x) + k*(s_val - x)
            dy = x*(rho - z) - y
            dz = x*y - beta*z
            return np.array([dx, dy, dz], dtype=float)

        u_rx_next = rk4_step(rx_rhs, u_rx, t, dt)
        tx[i] = u_tx_next
        rx[i] = u_rx_next
        s[i]  = s_t
        u_tx = u_tx_next
        u_rx = u_rx_next
        t += dt

    return tx, rx, s

# Calculate the critical value of k
def compute_cle_from_trajs(tx_traj, rx_traj, k, dt, ortho_interval=200, t_transient=0.0):
    """
    tx_traj, rx_traj: arrays shape (N,3)
    Integrate V (3x3) via dV/dt = J(t) @ V with J built from tx_traj[i], rx_traj[i] and k.
    Perform QR orthonormalization every ortho_interval steps and accumulate log norms.
    Return vector of 3 Lyapunov exponents (largest first).
    """
    N = tx_traj.shape[0]
    start_idx = int(np.ceil(t_transient / dt))
    if start_idx >= N:
        raise ValueError("t_transient >= total time; reduce t_transient or increase simulation time.")
    V = np.eye(3, dtype=float)
    le_accum = np.zeros(3, dtype=float)
    count = 0
    for i in range(start_idx, N-1):
        J1 = jacobian_error_mixed(tx_traj[i], rx_traj[i], k)
        # K1 = J1 @ V
        K1 = J1 @ V
        mid = min(i+1, N-1)
        J2 = jacobian_error_mixed(tx_traj[mid], rx_traj[mid], k)
        K2 = J2 @ (V + 0.5*dt*K1)
        K3 = J2 @ (V + 0.5*dt*K2)
        nxt = min(i+1, N-1)
        J4 = jacobian_error_mixed(tx_traj[nxt], rx_traj[nxt], k)
        K4 = J4 @ (V + dt*K3)
        V = V + (dt/6.0)*(K1 + 2*K2 + 2*K3 + K4)
        step_idx = i - start_idx + 1
        if (step_idx % ortho_interval) == 0:
            Q, R = np.linalg.qr(V)
            le_accum += np.log(np.abs(np.diag(R)))
            count += 1
            V = Q 

    if count == 0:
        raise RuntimeError("No orthonormalization performed: increase simulation duration or decrease ortho_interval")

    total_time = count * ortho_interval * dt
    lyap = le_accum / total_time
    idx_sort = np.argsort(-lyap)
    return lyap[idx_sort], lyap


dt_sweep = 0.001  
T_sweep = 50.0 
transient = 10.0 

k_values = np.linspace(0, 50, 21) 

u0_tx = np.array([0.1, 1.0, 1.05])
u0_rx = np.array([5.0, -5.0, 10.0])

N_sweep = int(np.ceil(T_sweep / dt_sweep))
m_zeros = np.zeros(N_sweep)

max_lyaps = []

print(f"Starting k_c sweep (from {k_values[0]} to {k_values[-1]})...")

for k_val in k_values:
    tx, rx, s = simulate_tx_rx_with_message(
        u0_tx, u0_rx, 
        m_zeros, 
        alpha=0.0,
        k=k_val, 
        dt=dt_sweep
    )
    sorted_lyap, _ = compute_cle_from_trajs(
        tx, rx, k_val, dt_sweep, 
        ortho_interval=100, 
        t_transient=transient
    )
    lambda_1 = sorted_lyap[0]
    max_lyaps.append(lambda_1)
    
    print(f"  k = {k_val:.2f}, Max CLE = {lambda_1:.4f}")

max_lyaps = np.array(max_lyaps)
kc_estimate = None
for i in range(len(max_lyaps) - 1):
    if max_lyaps[i] > 0 and max_lyaps[i+1] < 0:
        slope = (max_lyaps[i+1] - max_lyaps[i]) / (k_values[i+1] - k_values[i])
        kc_estimate = k_values[i] - max_lyaps[i] / slope
        break

print("\n" + "="*40)
if kc_estimate:
    print(f"Calculated Critical Threshold k_c ≈ {kc_estimate:.4f}")
else:
    print("Zero crossing not found in this range. Try expanding the k_values range.")
print("="*40)


In [ ]:
import soundfile as sf
import speech_recognition as speech_rec
from scipy import signal

r = speech_rec.Recognizer()
dt = 0.0000625
T_sim = 10.0
N = int(np.ceil(T_sim / dt))
raw_audios, raw_labels, sr, factor = load_multiple_speechcommands_messages(
    label=None,
    target_rate=int(1/dt),
    repeat_to_len=N,
    num_samples=20,
)
audios_list = []
labels_list = []
for a, l in zip(raw_audios, raw_labels):
    if l != "unknown":
        audios_list.append(a)
        labels_list.append(l)
    if len(audios_list) == 10:
        break

num_msgs = len(audios_list)
print(f"Loaded {num_msgs} clean messages.\nLabels = {labels_list}")
u0_tx = np.array([0.1, 1.0, 1.05])
u0_rx = np.array([5.0, -5.0, 10.0])

alpha_list = np.arange(0.1, 0.55, 0.05)
k_list     = np.arange(100, 505, 5)
# alpha_list = np.array([0.5])
# k_list     = np.array([500])
transient_idx = int(0.5 / dt)

results_log = []
correct_count = 0 

for msg_idx, (m_ds, label) in enumerate(zip(audios_list, labels_list)):

    print(f"\n==============================")
    print(f"Processing Message {msg_idx+1}/{num_msgs}, label = {label}")
    print(f"==============================")

    best_corr = -1
    best_mse = float('inf') 
    best_pair = (None, None)
    best_m_hat = None

    for alpha in alpha_list:
        for k in k_list:
            tx, rx, s = simulate_tx_rx_with_message(u0_tx, u0_rx, m_ds, alpha, k, dt)
            x2 = rx[:,0]
            m_hat_tmp = (s - x2) / alpha
            curr_mse  = np.mean((m_hat_tmp[transient_idx:] - m_ds[transient_idx:])**2)
            curr_corr = np.corrcoef(m_hat_tmp[transient_idx:], m_ds[transient_idx:])[0,1]
            if curr_corr > best_corr:
                best_corr = curr_corr
                best_mse  = curr_mse    
                best_pair = (alpha, k)
                best_m_hat = m_hat_tmp

    print(f"Best: α={best_pair[0]:.2f}, k={best_pair[1]}, Corr={best_corr:.4f}, MSE={best_mse:.2e}")
    seg = slice(0, len(m_ds))

    plt.figure(figsize=(12,8))
    plt.subplot(3,1,1)
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, m_ds[seg], label="m (Original)")
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, best_m_hat[seg], '--', label="m_hat (Recovered)")
    plt.legend()
    plt.title(f"Message {msg_idx+1}, label={label}")
    plt.subplot(3,1,2)
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, best_m_hat[seg],
             '--', color='orange', linewidth=3, label="m_hat")
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, m_ds[seg],
             color='tab:blue', alpha=0.7, label="m")
    plt.legend(loc='upper right')
    plt.title(f"Message {msg_idx+1}, label={label} (Blue on Top)")
    x1 = tx[:,0]
    x2 = rx[:,0]
    plt.subplot(3,1,3)
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, x1[seg], label='x1 (TX)')
    plt.plot(seg.start*dt + np.arange(seg.stop-seg.start)*dt, x2[seg], label='x2 (RX)', alpha=0.8)
    plt.legend()
    plt.title("TX vs RX (x coordinate)")

    plt.tight_layout()
    plt.show()

    # ASR Verification 
    filename_orig = f"temp_orig_{msg_idx}.wav"
    filename_hat  = f"temp_hat_{msg_idx}.wav"
    sf.write(filename_orig, m_ds, samplerate=int(1/dt))
    sf.write(filename_hat, best_m_hat, samplerate=int(1/dt))

    text_hat = "Recognition Failed"
    
    try:
        with speech_rec.AudioFile(filename_hat) as source:
            audio_data = r.record(source)
            text_hat = r.recognize_google(audio_data)
    except speech_rec.UnknownValueError:
        text_hat = "[Unintelligible]"
    except speech_rec.RequestError:
        text_hat = "[API Error]"
    except Exception as e:
        text_hat = f"[Error: {e}]"
    clean_label = str(label).lower().strip()
    clean_recog = str(text_hat).lower().strip()
    is_correct = clean_label in clean_recog 
    
    if is_correct:
        correct_count += 1
        match_str = "YES"
    else:
        match_str = "NO"

    print(f"  -> Original: {clean_label} | Recog: {clean_recog} | Match: {match_str}")
    results_log.append({
        "id": msg_idx + 1,
        "label": label,
        "alpha": best_pair[0],
        "k": best_pair[1],
        "mse": best_mse,
        "corr": best_corr,
        "recog_text": text_hat,
        "match": is_correct
    })

print("\n\n" + "="*60)
print(f"{'ID':<4} {'Label':<10} {'Corr':<8} {'MSE':<10} {'Recognized':<15} {'Match'}")
print("-" * 60)

total_mse = 0
total_corr = 0

for res in results_log:
    total_mse += res['mse']
    total_corr += res['corr']
    match_mark = "✔" if res['match'] else "✘"
    print(f"{res['id']:<4} {res['label']:<10} {res['corr']:.4f}   {res['mse']:.2e}   {res['recog_text']:<15} {match_mark}")

avg_mse = total_mse / num_msgs
avg_corr = total_corr / num_msgs
accuracy = (correct_count / num_msgs) * 100

print("="*60)
print(f"Total Messages Processed: {num_msgs}")
print(f"Average Correlation:      {avg_corr:.4f}")
print(f"Average MSE:              {avg_mse:.4e}")
print(f"ASR Accuracy:             {accuracy:.2f}% ({correct_count}/{num_msgs})")
print("="*60)